<center>
    <p>115 Theoretical Neuroscience II</p>
    <h1></h1>
    <h1>Lecture 04:</h1>
    <h1>Associative memory</h1>
    <p>----</p>
    <p>Prof. Jochen Braun Ph.D.</p>
    <p>Institute of Biology</p>
    <p>Otto-von-Guericke University Magdeburg</p>
    <p>----</p>
    <p>Textbook:</p>
    <p>Peter Dayan & Larry Abbott (2001) Theoretical Neuroscience, MIT Press.</p>
</center>

In [ ]:
from IPython.display import display, Math

# Inject custom MathJax configuration for additional packages
custom_latex = r"""
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage{eufrak}
"""

# **Abstract:**

A recurrent network may store several 'memory' patterns in its connectivity (synaptic weights).  **Memory patterns** are steady-state patterns of activity.  Given any input pattern, the network converges to the most similar steady-state ("long-term memory").  Additionally, memory patterns may remain active after input has ceased ("short-term" or "working memory").

Today we construct an associative memory network.  To encode memory patterns, we must ensure that excitation spreads preferentially within the pattern (rather than 'leaks out' to other patterns).  To this end, we choose **connection weights** such that arbitrary memory patterns become eigenvectors of connectivity.  Additionally, we must ensure that memory patterns are steady-states.  To this end, we **fine-tune the activation function** of units.  Thus, functionality depends on properties of both network  (connectivity) and units (activation function).

We visualize associative memory behaviour in terms of **potential landscapes**, with local minima representing memory patterns.



# **Overview:**

**1. Associative memory**

**2. Associative memory network**

**3. Inscribing memory patterns into connectivity**

**4. Steady-state condition**

**5. Simulated dynamics**

**6. Appendix:** Theoretical justification of potential landscape

<br>

**Credits:**

D&A, Chapter 7, Network models

# Activity patterns in vector space

An $n$-dimensional pattern of activity $\{ x_1, x_2, \ldots , x_n\}$ can be represented as a point in vector space $\mathfrak{R}^n$. Different patterns correspond to different points in vector space.  Similar patterns map to nearby points, dissimilar patterns to distant points.


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec03_ActivityPatternVector.png" width="600">

(Slide-1)

# **1. Associative memory**

A recurrent network may function as an "associative" or "content-addressable" memory.  Such a network may store several patterns ('memory patterns') and retrieve these patterns when prompted by input.  In particular, it may retrieve the one pattern among all stored patterns that is most similar to the input.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Memory_patterns.png" width="600">

(Slide-2)

# Selective amplification

In the last lecture, we have seen that recurrent networks can selectively amplify a particular pattern that is stored in its connectivity.  We have also seen that the amplification factor will depend on the degree of similarity between input and stored pattern, specifically, on the projection of the input onto the stored pattern.


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Schematic_a.png" width="600">

(Slide-3)

# Amplifying arbitrary patterns

Today we consider networks that store several arbitrary patterns, which we call memory patterns.  When challenged with an input, we would like the network to selectively amplify the one memory pattern that most resembles the input.

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Schematic_b.png" width="600">

(Slide-4)

# Memory and persistent activity

Associative memory networks model model mammalian "long-term memory" systems that are characterized by recurrent connections, such as prefrontal cortex or the CA areas of hippocampus.

Note that long-term memory patterns are stored in terms of connectivity matrices (synaptic weights), not in terms of activity.   However, we will see that memory patterns are also steady-state patterns of network activity.

Depending on details, memory patterns may remain active even after input has ceased ("persistent activity").  Such persistent activity models "short-term" or "working memory".

(Slide-5)

# Potential landscape

Conceptually, we can imagine all possible activity patterns as points on a hyperplane and assign a 'potential energy' to each pattern/point.  For purposes of visualization, we restrict ourselves to a two-dimensional plane. The 'potential energy' is chosen in such a way that its **gradient** corresponds to the average **speed and direction** of activity changes, when the network is allowed to 'reverberate' (develop freely). **Local minima** in 'potential energy' represent stable steady-state patterns of activity.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Landscape.png" width="600">

The process associative memory retrieval can then be conceptualized as follows: (i) initialize the network to a particular pattern (point on surface), (ii) let the network 'reverberate' (relax) and allow activity to follow the gradient to the nearest 'potential well', (iii) read out the memory pattern at the potential mininum.

Potential minima are typically surrounded by an area called 'basin of attraction'. Within this area, network activity is almost certain to develop towards this particular minimum.  Outside this area, the development is likely to take a different direction.

(Slide-6)

# Goal of Lecture


- The goal of this lecture is to construct a non-linear recurrent network that functions as an associative memory.  


- Specifically, we wish to store several (possibly overlapping) memory patterns in the connectivity matrix.  


- We further wish the network to settle, after being challenged with an arbitrary input, to the memory pattern that most resembles the input.


- We will discover that associative memory is a fragile feature, requiring delicate calibration of connectivity and responsiveness.

(Slide-7)


In [ ]:
# @title Associative memory simulation {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import matplotlib.pyplot as plt

def MakeMemoryPattern(N, alpha):
    # length N, sparseness alpha
    # number on
    nc = round(alpha * N)
    # number off
    n0 = N - nc
    # memory vector
    vm = np.concatenate((np.ones(nc), np.zeros(n0)))
    # randomize order
    np.random.shuffle(vm)
    return vm

def AutoCovarianceMatrix(vm, N, alpha):
    # pattern vm, length N, sparseness alpha
    v0 = vm - alpha * np.ones(N)
    return np.outer(v0, v0)

def ActivationFunction(vin, rmax, theta):
    # input vin, saturation rmax, threshold theta
    vout = rmax * np.tanh((vin - theta) / rmax)
    vout = np.maximum(vout, 0)
    return vout

def ConnectivityMatrix( N, Nmem, alpha, kappa):
    # size N, memories Nmem, sparseness alpha, recurrent excitation kappa
    Vm = np.empty((Nmem, N))
    K = np.zeros((N, N))
    Cm_list = []

    for m in range(Nmem):
        vm = MakeMemoryPattern(N, alpha)
        Vm[m, :] = vm
        Cm = AutoCovarianceMatrix(vm, N, alpha)
        Cm_list.append(Cm)
        K += Cm

    K = K / ( alpha * (1 - alpha) * N )
    M = kappa * K - 1 / (alpha * N)

    return Vm, K, M, Cm_list

def CheckMemoryVectors( Vm, alpha ):

    Vm_sum  = np.sum(Vm, axis=1)
    Vm_norm  = np.linalg.norm(Vm, axis=1)
    Vmz_sum = np.sum(Vm - alpha, axis=1)
    Vmz_norm = np.linalg.norm(Vm - alpha, axis=1)

    print(f'Vm_sum = {Vm_sum}')
    print(f'Vm_norm = {Vm_norm}')
    print(f'Vmz_sum = {Vmz_sum}')
    print(f'Vmz_norm = {Vmz_norm}')

# Compute A matrix
#A = np.zeros((Nmem, Nmem))
#for m in range(Nmem):
#    vm = Vm[m, :]
#    Kvm = K @ vm
#    vma = vm - alpha

#    for n in range(Nmem):
#        vn = Vm[n, :]
#        vna = vn - alpha
#        A[m, n] = (Kvm @ vna) / (Vmz_len[m] * Vmz_len[n])

# Check dot product of memory vectors with K
def DotProductWithK( Vm, K, alpha ):

    Nmem = Vm.shape[0]
    N = Vm.shape[1]
    B = np.zeros((Nmem, Nmem))
    for m in range(Nmem):
        vm = Vm[m, :]
        Kvm = K @ vm
        for n in range(Nmem):
            vn = Vm[n, :]
            vna = vn - alpha
            B[m, n] = Kvm @ vna
    return B

# Check dot product of memory vectors with M
def DotProductWithM( Vm, M, alpha, kappa ):

    Nmem = Vm.shape[0]
    N = Vm.shape[1]
    C = np.zeros((Nmem, Nmem))
    for m in range(Nmem):
        vm = Vm[m, :]
        Mvm = M @ vm
        for n in range(Nmem):
            vn = Vm[n, :]
            vz = kappa * (vn - alpha) - 1
            vz_norm = np.linalg.norm(vz)
            C[m, n] = np.linalg.norm(Mvm - vz) / vz_norm

    return C

# Check trajectory vectors
def CheckTrajectoryVectors( Vm, M, alpha, rmax, theta ):

    Nmem = Vm.shape[0]
    N = Vm.shape[1]
    D = np.zeros((Nmem, Nmem))
    c = 50;

    for m in range(Nmem):
        vm = Vm[m, :]
        Mvm = M @ vm
        for n in range(Nmem):
            vn = Vm[n, :]

            Dvnm = - (c*vn) + ActivationFunction( c * Mvm, rmax, theta )

            Dvnm_norm = np.norm( Dvnm, axis=1 );

            # Dvnm should be short for n=m, long otherwise

            D[m,n] = Dvnm_norm;

    return D

# Simulate dynamics
def SimulateDynamics( N, Nmem, Vm, M, tau, dt, Tend, scalefactor, rmax, theta, alpha, kappa ):

    # time axis
    ti = np.linspace(0, Tend, int(Tend / dt))

    # concatenate memory patterns and control patterns
    Fm = np.empty((2 * Nmem, N))
    Fm[:Nmem, :] = Vm
    for m in range(Nmem):
        Fm[Nmem + m, :] = MakeMemoryPattern(N, alpha)

    # initial condition as linear combination of two random memory patterns
    k1, k2 = np.random.randint(0, Nmem, 2)
    c1, c2 = scalefactor, 0
    vold   = c1 * Fm[k1, :] + c2 * Fm[k2, :]

    # projections of initial condition
    P0 = np.empty((1, 2 * Nmem))
    for m in range(2 * Nmem):
            fm = Fm[m, :]
            P0[0, m] = np.dot(vold, fm) / np.linalg.norm(fm)

    # allocate matrices for projections and for patterns
    Pi = np.empty((len(ti), 2 * Nmem))
    Vi = np.empty((len(ti), N))

    # iteration
    for i in range(len(ti)):
        vin = M @ vold
        vinf = ActivationFunction(vin, rmax, theta)
        vnew = vinf + (vold - vinf) * np.exp(-dt / tau)
        # new pattern
        Vi[i, :] = vnew
        vold = vnew
        # projections
        for m in range(2 * Nmem):
            fm = Fm[m, :]
            Pi[i, m] = np.dot(vnew, fm) / np.linalg.norm(fm)

    return ti, P0, Pi, Vi


def PlotPatternDynamic( plt, axs, ti, Vm, Vi, Pi ):

    N = Vm.shape[1]
    Nmem = Vm.shape[0]

    Tend = ti[-1]

    # Define parameters
    clr = np.array([[0.25, 0.75, 0.25],
                   [1.00, 0.25, 0.25],
                   [1.00, 0.85, 0.15],
                   [0.25, 0.25, 1.00]])

    fs = 16

    # ellipses to show memory patterns
    angle = np.linspace(-np.pi, np.pi, 20)
    sx = np.cos(angle)
    sy = np.sin(angle)

    # Select the three dominant patterns (those with the largest projections)
    Y = np.sort(Pi[-1, 0:Nmem])
    kidx = np.argsort(Pi[-1, 0:Nmem])
    Nkk = 3
    kk = kidx[-Nkk:]

    # white background
    axs.fill([0, Tend, Tend, 0, 0], [0, 0, N, N, 0], 'w')

    # Loop over dominant patterns, with the largest pattern last
    for k in range(Nkk):
        ki = kk[k]
        kclr = clr[ki,:]
        nn = np.where(Vm[ki, :] > 0)[0]

        for n in nn:
            tii = np.concatenate((ti, ti[::-1]))
            Vii = np.concatenate((Vi[:, n], -Vi[::-1, n])) / np.max(Vi)

            axs.fill(tii, n + 0.5 + Vii, color=kclr, linewidth=0.5, edgecolor='k')

            yfactor = Vm[ki, n] / np.max(Vm)
            xfactor = 0.7 * yfactor * Tend / N
            axs.fill(-0.5 - 0.5 * k + sx * xfactor, n + 0.5 + sy * yfactor, color=kclr, edgecolor='k')

    axs.set_xlim([-0.5 * Nkk - 0.5, Tend])
    axs.set_ylim([-0.5, N+0.5])
    axs.set_xlabel('time', fontsize=fs)
    axs.set_ylabel('unit number', fontsize=fs)
    axs.set_xticks(np.arange(0,Tend))
    axs.set_yticks(np.arange(0,N,10))
    axs.set_title('Dynamic of unit activity, relative to global maximum')


def PlotPatternNormalizedDynamic( plt, axs, ti, Vm, Vi, Pi ):

    N = Vm.shape[1]
    Nmem = Vm.shape[0]

    Tend = ti[-1]

    # Define parameters
    clr = np.array([[0.25, 0.75, 0.25],
                   [1.00, 0.25, 0.25],
                   [1.00, 0.85, 0.15],
                   [0.25, 0.25, 1.00]])

    fs = 16

    # ellipses to show memory patterns
    angle = np.linspace(-np.pi, np.pi, 20)
    sx = np.cos(angle)
    sy = np.sin(angle)

    # Select the three dominant patterns (those with the largest projections)
    Y = np.sort(Pi[-1, 0:Nmem])
    kidx = np.argsort(Pi[-1, 0:Nmem])
    Nkk = 3
    kk = kidx[-Nkk:]

    # white background
    axs.fill([0, Tend, Tend, 0, 0], [0, 0, N, N, 0], 'w')

    # Loop over dominant patterns, with the largest pattern last
    for k in range(Nkk):
        ki = kk[k]
        kclr = clr[ki,:]
        nn = np.where(Vm[ki, :] > 0)[0]

        for n in nn:
            tii = np.concatenate((ti, ti[::-1]))
            Vii = np.concatenate((Vi[:, n], -Vi[::-1, n])) / np.max(Vi[:,n])

            axs.fill(tii, n + 0.5 + Vii, color=kclr, linewidth=0.5, edgecolor='k')

            yfactor = Vm[ki, n] / np.max(Vm)
            xfactor = 0.7 * yfactor * Tend / N
            axs.fill(-0.5 - 0.5 * k + sx * xfactor, n + 0.5 + sy * yfactor, color=kclr, edgecolor='k')

    axs.set_xlim([-0.5 * Nkk - 0.5, Tend])
    axs.set_ylim([-0.5, N+0.5])
    axs.set_xlabel('time', fontsize=fs)
    axs.set_ylabel('unit number', fontsize=fs)
    axs.set_xticks(np.arange(0,Tend))
    axs.set_yticks(np.arange(0,N,10))
    axs.set_title('Dynamic of unit activity, relative to unit maximum')

def PlotProjectionDynamic( axs, Nmem, ti, P0, Pi):

    clr = np.array([[0.25, 0.75, 0.25],
                   [1.00, 0.25, 0.25],
                   [1.00, 0.85, 0.15],
                   [0.25, 0.25, 1.00]])

    for m in range(Nmem):
        axs.plot(ti, Pi[:, m], color=clr[m,:], label=f"v {m}")
        axs.plot(0, P0[0, m], 'o', color=clr[m,:], markersize=10)

    axs.set_xlabel('Time')
    axs.set_ylabel('Projection')
    axs.set_title('Projection of network activity onto memory patterns')

    axs.legend(loc='lower right')

def plot_associative_memory_simulation(scalefactor=2, format=1):
    """(Slide-8)"""

    # Network parameters
    N = 50
    tau = 1
    rmax = 150
    theta = -20
    Nmem = 4
    alpha = 0.2
    kappa = 2.0

    deltat = 0.02
    Tend = 5

    # make connectivity matrix
    (Vm, K, M, Cm_list) = ConnectivityMatrix( N, Nmem, alpha, kappa)

    # simulate dynamics
    # scaling of initial condition
    (ti, P0, Pi, Vi) = SimulateDynamics( N, Nmem, Vm, M, tau, deltat, Tend, scalefactor, rmax, theta, alpha, kappa )

    # Plot results
    fig, axs = plt.subplots( 1,1, figsize=(8,6))

    if format == 0:
        PlotProjectionDynamic( axs, Nmem, ti, P0, Pi)
    else:
        if format == 1:
            PlotPatternDynamic( fig, axs, ti, Vm, Vi, Pi )
        else:
            PlotPatternNormalizedDynamic( fig, axs, ti, Vm, Vi, Pi )


    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_associative_memory_simulation, scalefactor = (1.0, 20.0, 1.0 ), format=(0,2,1) )


interactive(children=(FloatSlider(value=2.0, description='scalefactor', max=20.0, min=1.0, step=1.0), IntSlide…

<function __main__.plot_associative_memory_simulation(scalefactor=2, format=1)>

# **2. Asssociative memory network**

We consider a non-linear recurrent network

$$
\tau \frac{d{\bf v}}{dt} = -{\bf v} + {\bf F}\left[{\bf M} \cdot {\bf v}\right]
$$

with time-varying activity ${\bf v}(t)$, connectivity matrix $\bf M$, and relaxation time $\tau$.  Instead of a constant external input, we prompt the network with some initial activity pattern ${\bf v}(0)$.

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Schematic_b.png" width="600">

(Slide-9)

# Sigmoidal non-linearity

The activation function is crucial for associative memory function. We use a sigmoidal function with threshold

$$
F(v) = r_{max} \, \left[ \tanh \left( \frac{v-\vartheta}{r_{max}} \right) \right]_+
$$

and (with the benefit of hindsight) choose the parameters

$$\begin{eqnarray}
r_{max} = 150 Hz \qquad & \rm{maximal \, activity}\\
\vartheta = -20Hz\qquad & \rm{background \, activity}
\end{eqnarray}$$


<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_a.png" width="600">

Note: Negative $\vartheta$ adds to activity $v$ (acting like background activity).  Positive $\vartheta$ would subtract from activity $v$ (acting like threshold).

<br>

(Slide-10)






# Memory patterns

We need to provide our network with many more units

$$\begin{eqnarray}
N = 50 \qquad & \rm{number \, of \, units}
\end{eqnarray}$$

than stored memory patterns

$$
N_{mem}=4 \qquad  \rm{stored \, patterns}
$$

In other words, we must ensure $N_{mem} << N$.

For simplicity, we consider patterns with $\alpha N$  components of value $1$ and $(1-\alpha) N$ components of value $0$.  An example for such a pattern is
$$
{\bf v}_1 = \left[ 1, \quad \ldots, \quad 1, \quad  0, \quad \ldots, \quad 0\right]
$$


Sparseness $\alpha$ (fraction of active units) is another critical paramemter. We choose

$$\begin{eqnarray}
\alpha = 0.2 \qquad & \rm{sparseness}\\
\alpha N = 10 \qquad & \textrm{'on' units}\\
(1-\alpha) N = 40 \qquad & \textrm{'off' units}
\end{eqnarray}$$

<br>

(Slide-11)

# Four example patterns

Here is an illustration of $N_{mem}=4$ patterns with $\alpha N=10$ ones and $(1-\alpha)N=40$ zeros. From each element, $\alpha$ has been subtracted, so that the final values are either $1-\alpha$ (large red squares) or $-\alpha$ (small green squares).


<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_b.png" width="600">


<br>

Note that the sum of all components is always
$$
\mathit{trace}({\bf v}_m) = \sum_a v_m^a = \alpha N, \qquad\qquad \mathit{trace}({\bf v}_m - \alpha {\bf n}) = \sum_a (v_m^a - \alpha) = 0
$$

(Slide-12)



# Uniform patterns

In addition, we will use uniform patterns such as

$$
{\bf n} \equiv \left[ 1, \quad \ldots, \quad 1  \right]
$$

and

$$
\bar {\bf v}\equiv  \left[ \alpha, \quad \ldots, \quad \alpha  \right] = \alpha {\bf n}
$$

The components of $\bar {\bf v}$ sum to the same value as the memory patterns

$$
\mathit{trace}(\bar {\bf v}) = \sum_a
{r v}_a = \alpha N
$$

<br>

(Slide-13)



# Dot products

With these definitions, we can form the following dot products:

$$
{\bf n} \cdot {\bf v}_m = \alpha\,N
$$

<br>

$$
{\bf v}_m \cdot {\bf v}_m  \quad =
\alpha \, N  \qquad \forall m
$$

<vr>

$$
{\bf v}_n \cdot {\bf v}_m  \quad \approx  \alpha^2 \, N \pm \sqrt{\alpha^2 (1-\alpha^2) N }\qquad \mathrm{for} \quad n \neq m
$$

<br>

Note that $\alpha^2$ is the probability that corresponding elements of ${\bf v}_n$ and ${\bf v}_m$ are both positive.

(Slide-14)

# Stability condition

If initial activity ${\bf v}_0$ is proportional to ${\bf v}_m$, we want activity to develop towards ${\bf v}_m$. In other words, we want memory patterns ${\bf v}_m$ to be stable steady-states.  

$$
{\bf v}_0 \propto {\bf v}_m \qquad\Rightarrow\qquad {\bf v}(t)\rightarrow {\bf v}_m
\qquad\Rightarrow\qquad
0  = -{\bf v}_m +  F\left({\bf M} \cdot {\bf v}_m\right)
$$

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Stability_b.png" width="400">

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Stability_a.png" width="400">

Left: emergence of a stable pattern (green).  Right: projection of network activity on different pattern vectors (blue, red, green, etc).

(Slide-15)

# Summary so far


We have now defined our objective: a recurrent network with certain, predefined memory patterns as stable steady-states.


Next we need to construct a suitable connectivity matrix.


Although the principle is simple, the details are fiddly ...

(Slide-16)


# **3. Inscribing memory patterns into connectivity**

Our basic intuition is that a memory pattern will be stable if the excitation propagates within active nodes, but does not 'leak out' to inactive nodes.  To this end, the connections between active nodes should be strong, but connections between active and inactive nodes weak.

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Hippocampus.jpg" width="800">

<br>

Conceptual cartoon of associative memory (from hippocampus lecture).

(Slide-17)

# Sum of covariance matrices

To find a suitable connection matrix ${\bf M}$, we first consider the sum ${\bf K}$ of the covariance matrices of the to-be-stored patterns ${\bf v}_m$:

$$
{\bf K} = \frac{1}{\alpha (1-\alpha) N} \, \sum_m \, {\bf C}^m
$$


Covariance matrix ($=$ product of deviations from mean)

$$
{\bf C}^m = \left({\bf v}_m - \alpha \, {\bf n} \right) \left({\bf v}_m - \alpha \, {\bf n} \right)
$$


By inscribing the **covariance** between active members of a memory pattern into the connectivity, we ensure that excitation spreads preferentially  **within** the pattern (rather than 'leaking out' to other patterns).   Of course, any overlap between different memory patterns poses a problem, because any active member that also belongs to another pattern constitutes a 'leak'.

(Slide-18)

# First pattern covariance

Illustration of $N_{mem}=4$ patterns $(\bf v_m-\alpha \bf n)$ (left) and of first covariance matrix $\bf C_1=(\bf v_1-\alpha \bf n)(\bf v_1-\alpha \bf n)$ (right).  Red squares indicate positive, green squares negative values.  Size indicates absolute value.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_c.png" width="600">

(Slide-19)

#  Second pattern covariance

Illustration of $N_{mem}=4$ patterns $(\bf v_m-\alpha \bf n)$ (left) and of second covariance matrix $\bf C_2=(\bf v_2-\alpha \bf n)(\bf v_2-\alpha \bf n)$ (right).  

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_d.png" width="600">

(Slide-20)

# Third pattern covariance

Illustration of $N_{mem}=4$ patterns (left) and of third covariance matrix $\bf C_3=(\bf v_3-\alpha \bf n)(\bf v_3-\alpha \bf n)$ (right).  

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_e.png" width="600">

(Slide-21)

#  Fourth pattern covariance

Illustration of $N_{mem}=4$ patterns (left) and of fourth covariance matrix $\bf C_4=(\bf v_4-\alpha \bf n)(\bf v_4-\alpha \bf n)$ (right).  

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_f.png" width="600">

(Slide-22)

# Four covariances superimposed


Sum of all four covariance matrices (right).  Red squares indicate positive, green squares negative values.  Size indicates absolute value.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_g.png" width="600">

(Slide-23)

# Propagation of activity

To analyze propagation of memory pattern ${\bf v}_n$ through matrix ${\bf K}$, we consider the dot product between ${\bf K}$ and ${\bf v}_n$

$$
{\bf K} \cdot {\bf v}_n  = \frac{1}{\alpha (1-\alpha) N} \, \sum_m \, {\bf C}^m \cdot {\bf v}_n
$$

which describes the outcome of one propagation step.

<br>

As it turns out, the outcome approximates (see proof below) a linear combination of the memory pattern ${\bf v}_n$ and uniform pattern ${\bf n}$:

$$
{\bf K} \cdot {\bf v}_n \approx {\bf v}_n -  \alpha \, {\bf n}
$$

While this may not be exactly what we hoped for, we can work with this!

(Slide-24)

# Proof

$$\begin{eqnarray}
{\bf C}_m \cdot {\bf v}_n &=& \left({\bf v}_m - \alpha \,  {\bf n} \right) \left[ \left({\bf v}_m - \alpha  \, {\bf n} \right) \cdot {\bf v}_n \right] =
\\
\\
&=& \left({\bf v}_m - \alpha \,  {\bf n} \right) \left[ {\bf v}_m \cdot {\bf v}_n -  \alpha^2 N\right] =
\\
\\
&=& \left({\bf v}_m - \alpha \, {\bf n} \right) \left\{
\begin{array}{cc}
\left[  \alpha N - \alpha^2 N\right] & {\rm for} \, n=m
\\
\\
\left[ \alpha^2 N \pm \sqrt{\alpha^2 (1-\alpha^2) N }-  \alpha^2 N\right] & {\rm for} \, n\neq m
\end{array}
\right.
\\
\\
&=&  \alpha (1-\alpha) N \, \left({\bf v}_m - \alpha \,  {\bf n} \right)
\left\{
\begin{array}{cc}
1 & {\rm for} \, n=m
\\
\\
\pm \sqrt{\frac{ 1-\alpha^2}{(1-\alpha)^2 \, N} } & {\rm for} \, n\neq m
\end{array}
\right.
\end{eqnarray}$$

<br>

Thus, we see that $\frac{1}{\alpha (1-\alpha) N} \, {\bf C}_m \cdot {\bf v}_n$ is either $\left({\bf v}_m - \alpha \,  {\bf n} \right)$ or near zero, depending on whether $n=m$ or $n\neq m$, respectively.  Accordingly, we conclude that

$$\begin{eqnarray}
{\bf K} \cdot {\bf v}_n &=& \frac{1}{\alpha (1-\alpha) N} \, \sum_m \, {\bf C}_m \cdot {\bf v}_n
\\
&\approx& {\bf v}_n -  \alpha\, {\bf n}
\end{eqnarray}$$

QED

(Slide-25)

# Once again



To repeat this important point in a slightly different way, the dot product between ${\bf K}$ and memory pattern ${\bf v_1}$ approximates a linear combination of ${\bf v_1}$ and ${\bf n}$:

$$
{\bf K} = \frac{1}{\alpha (1-\alpha) N} \, \sum_m \, {\bf C}_m
$$

$$\begin{eqnarray}
{\bf K} \cdot {\bf v}_1 &=& \frac{1}{\alpha (1-\alpha) N} \,\left[ {\bf C}_1 \cdot {\bf v}_1 + {\bf C}_2 \cdot {\bf v}_1 + \ldots + {\bf C}_m \cdot {\bf v}_1\right] =
\\
\\
&\approx& \frac{1}{\alpha (1-\alpha) N} \, \left[  \alpha(1-\alpha)N\left({\bf v}_1 -  \alpha \, {\bf n} \right) + 0 + \ldots + 0\right]
\\
\\
&=& {\bf v}_1 -  \alpha \, {\bf n}
\end{eqnarray}$$

(Slide-26)

# Covariance is balanced

Note that covariance matrices are balanced in that a uniform input ${\bf n}$ produces neither net excitation nor net inhibition:

$$\begin{eqnarray}
{\bf C}_m \cdot {\bf n} =&  \left[\begin{array}{ccc} \alpha^2 & \ldots & -(1-\alpha) \alpha \\ \vdots & \ddots & \vdots \\ -(1-\alpha)\alpha & \ldots &(1-\alpha)^2 \end{array}\right]
\cdot \left[ \begin{array}{c} 1 \\ \vdots \\ 1 \end{array}\right] =
\end{eqnarray}$$

$$
=  \alpha (1-\alpha) N \, \left[\begin{array}{c}
\alpha  - \alpha
\\
\vdots
\\
-(1-\alpha) + 1-\alpha
\end{array} \right] = \left[ \begin{array}{c}  0 \\ \vdots \\  0 \end{array} \right]
$$

The same is true for random input (random subset of elements active, others inactive).

(Slide-27)

# Dampening the spread of excitation with inhibition

The final connectivity matrix is obtained by scaling ${\bf K}$ with a factor $\kappa$ and by dampening the spread of excitation with uniform inhibition between all units:

$$
{\bf M} = \kappa \, {\bf K} - \frac{1}{\alpha N} \, {\bf n} \, {\bf n}
$$

The parameter $\kappa$ determines the relative strength of pattern-specific excitation.


The inhibition strength scales to unity when $\alpha N$ input units are active.


Note that the "vector outer product" ${\bf n} \, {\bf n}$ produces a matrix with all elements equal to unity.

(Slide-28)

# Final connectivity matrix

Here is the final connectivity matrix for the $N_{mem}=4$ patterns above.  Due to the pervasive inhibition, it contains mostly negative values (green squares). The memory patterns are still present however, because some connections are less negative than others.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Simulation_h.png" width="600">


(Slide-30)

# Summary so far


We have inscribed the desired memory patterns into connectivity: propagating a memory pattern (activity!) through the connection matrix, reproducing this pattern (as synaptic currents!).


To compensate for indiscriminate spreading of excitation, we have added some indiscriminate inhibition.


We now need to convince ourselves that the memory patterns (of activity) are indeed stable steady-states of the network.

Here, the details of the activation function will prove crucial.  

(Slide-31)

# **5. Steady-state condition**

A steady-state pattern of input activity ${\bf v}_\mathit{ss}$ must be balanced exactly by the pattern ${\bf M} \cdot {\bf v}_\mathit{ss}$ of synaptic currents:

$$
0 \stackrel{!}{=} \tau \frac{d {\bf v}}{dt}  = -{\bf v}_\mathit{ss} + F\left( {\bf M} \cdot {\bf v}_\mathit{ss}\right)
$$

If steady-state patterns of activity are scaled versions of memory patterns

$$
{\bf v}_\mathit{ss} = c \, {\bf v}_m
$$

they must satisfy the condition

$$
0 \stackrel{!}{=}  -c \, {\bf v}_m +  F\left( c \, {\bf M} \cdot {\bf v}_m\right)
$$

where $c$ is some scaling factor.

<br>

Thanks to our earlier hard work, we can make sense of this condition and rewrite it as follows:

$$\begin{eqnarray}
{\bf M} \cdot {\bf v}_m &=& \left[ \kappa \, {\bf K} - \frac{1}{\alpha\,N} {\bf n} \, {\bf n}\right] \cdot {\bf v}_m = \kappa \, {\bf K} \cdot  {\bf v}_m - \frac{\alpha N}{\alpha N} \, {\bf n} =
\\
\\
&=& \kappa \, \left({\bf v}_m -  \alpha \, {\bf n} \right)  -  {\bf n} =
\\
\\
&=& \kappa \, {\bf v}_m - (\kappa\alpha + 1) \, {\bf n} =
\\
\\
&=& \kappa \, {\bf v}_m - \gamma \, {\bf n},\qquad\qquad
\gamma \equiv \kappa \alpha + 1
\end{eqnarray}$$

The steady-state condition therefore becomes

$$\begin{eqnarray}
c \, {\bf v}_m &=& F\left[ c \left( {\bf M} \cdot {\bf v}_m\right)\right]
\\
\\
&=& F\left[ c \left( \kappa \, {\bf v}_m - \gamma \, {\bf n}\right)\right]
\end{eqnarray}$$

(Slide-32)


# Steady-state condition, continued

Taking ${\bf v}_1$ as an example and writing out the components

$$
c {\bf v}_1 = \left[
\begin{array}{c} c \\ \vdots \\ c \\ 0 \\ \vdots \\0 \end{array}
\right] = F\left( \kappa \, c \, {\bf v}_1 - \gamma \, c \,  {\bf n} \right) =
F\left(
\kappa \, \left[
\begin{array}{c} c \\ \vdots \\ c \\ 0 \\ \vdots \\0 \end{array}
\right] - \gamma \, \left[
\begin{array}{c} c \\ \vdots \\ c \\ c \\ \vdots \\ c \end{array}
\right]
\right)
$$

<br>

we obtain two steady-state conditions for scaling factor $c$, one from components $v_{1k}=1$:

$$
c = F(\kappa \, c - \gamma \, c)
$$

and another from components $v_{1k}=0$:

$$
0 = F(-\gamma \, c)
$$

(Slide-33)

# Activation function can meet conditions

Our activation function $F()$ meets both of these conditions

$$
F(v) = r_{max} \, \left[ \tanh \left( \frac{v-\vartheta}{r_{max}} \right) \right]_+,
\qquad
r_{max} = 150 Hz \qquad \vartheta = -20Hz
$$

provided the scaling factor $c$ takes a suitable value.

For parameter choices

$$
\alpha = 0.2, \qquad \kappa = 1.25, \qquad \gamma = \kappa \alpha + 1 = 1.25
$$

we obtain the following two conditions

$$
c = F[\kappa \, c - \gamma \, c] = F[0] \qquad \Rightarrow \qquad c  = 20 Hz
$$
$$
0 = F[-\gamma \, c] \qquad\qquad\qquad \Rightarrow \qquad c \geq 16 Hz
$$
since
$$
-\gamma \, c \leq \vartheta = -20Hz \qquad\Leftrightarrow \qquad c \geq \frac{20 Hz}{1.25} = 16 Hz
$$

<br>


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec04_Activation_function.png" width="600">

(Slide-34)

# Summary so far


We have convinced ourselves that the scaled memory patterns are indeed fixed points of the network.


Interestingly, the collective behaviour turned out to depend both on network features (connectivity) and on unit features (activation function).


Specifically, the activation function had to be just right (neither too steep, nor to flat) at input levels between zero activity and steady-state activity.


We can now validate this theoretical understanding of the collective dynamics by performing numerical simulations.

(Slide-35)

# **5. Simulated dynamics**

Finally, we are in a position to simulate the network dynamics:

$$
\tau \, \frac{d {\bf v}(t)}{dt} = - {\bf v}(t) + F\left[ {\bf M} \cdot {\bf v}(t) \right]
$$

Over a sufficiently small interval $\Delta t$, we can consider ${\bf v}(t)={\bf v}^{(i)}$ as constant and define

$$
{\bf v}_{ss}^{(i)} \equiv F\left[ {\bf M} \cdot {\bf v}^{(i)}\right]
$$

During this interval, the activity will relax from ${\bf v}^{(i)}$ towards ${\bf v}_{ss}^{(i)}$:

$$
{\bf v}^{(i+1)} = {\bf v}_{ss}^{(i)} + \left[ {\bf v}^{(i)} -  {\bf v}_{ss}^{(i)} \right] \, \exp\left( - \frac{\Delta t}{\tau} \right)
$$

(Slide-36)

# Projection on memory and non-memory patterns

Beginning with a combination of two memory patterns, we follow the evolution of network activity in terms of its projections (dot products) on $N_{mem}$ memory patterns and on an equal number of random control patterns.

Typically, one pattern emerges as dominant.  The others survive only to the extent that they overlap the dominant pattern.  In this simulation, $\kappa = 2$.

The 'scalefactor' sets the amplitude of initial activity ${\bf v}(0)$

In [ ]:
# @title Projection on memory and control patterns {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import matplotlib.pyplot as plt

def MakeMemoryPattern(N, alpha):
    # length N, sparseness alpha
    # number on
    nc = round(alpha * N)
    # number off
    n0 = N - nc
    # memory vector
    vm = np.concatenate((np.ones(nc), np.zeros(n0)))
    # randomize order
    np.random.shuffle(vm)
    return vm

def AutoCovarianceMatrix(vm, N, alpha):
    # pattern vm, length N, sparseness alpha
    v0 = vm - alpha * np.ones(N)
    return np.outer(v0, v0)

def ActivationFunction(vin, rmax, theta):
    # input vin, saturation rmax, threshold theta
    vout = rmax * np.tanh((vin - theta) / rmax)
    vout = np.maximum(vout, 0)
    return vout

def ConnectivityMatrix( N, Nmem, alpha, kappa):
    # size N, memories Nmem, sparseness alpha, recurrent excitation kappa
    Vm = np.empty((Nmem, N))
    K = np.zeros((N, N))
    Cm_list = []

    for m in range(Nmem):
        vm = MakeMemoryPattern(N, alpha)
        Vm[m, :] = vm
        Cm = AutoCovarianceMatrix(vm, N, alpha)
        Cm_list.append(Cm)
        K += Cm

    K = K / ( alpha * (1 - alpha) * N )
    M = kappa * K - 1 / (alpha * N)

    return Vm, K, M, Cm_list


# Simulate dynamics
def SimulateDynamics( N, Nmem, Vm, M, tau, dt, Tend, scalefactor, rmax, theta, alpha, kappa ):

    # time axis
    ti = np.linspace(0, Tend, int(Tend / dt))

    # concatenate memory patterns and control patterns
    Fm = np.empty((2 * Nmem, N))
    Fm[:Nmem, :] = Vm
    for m in range(Nmem):
        Fm[Nmem + m, :] = MakeMemoryPattern(N, alpha)

    # initial condition as linear combination of two random memory patterns
    k1, k2 = np.random.randint(0, Nmem, 2)
    c1, c2 = scalefactor, 0
    vold   = c1 * Fm[k1, :] + c2 * Fm[k2, :]

    # projections of initial condition
    P0 = np.empty((1, 2 * Nmem))
    for m in range(2 * Nmem):
            fm = Fm[m, :]
            P0[0, m] = np.dot(vold, fm) / np.linalg.norm(fm)

    # allocate matrices for projections and for patterns
    Pi = np.empty((len(ti), 2 * Nmem))
    Vi = np.empty((len(ti), N))

    # iteration
    for i in range(len(ti)):
        vin = M @ vold
        vinf = ActivationFunction(vin, rmax, theta)
        vnew = vinf + (vold - vinf) * np.exp(-dt / tau)
        # new pattern
        Vi[i, :] = vnew
        vold = vnew
        # projections
        for m in range(2 * Nmem):
            fm = Fm[m, :]
            Pi[i, m] = np.dot(vnew, fm) / np.linalg.norm(fm)

    return ti, P0, Pi, Vi

def PlotProjectionDynamicWithControl( axs, Nmem, ti, P0, Pi):

    clr = np.array([[0.25, 0.75, 0.25],
                   [0.75, 0.25, 0.25],
                   [0.85, 0.85, 0.15],
                   [0.25, 0.25, 0.75]])

    for m in range(Nmem):
        axs.plot(ti, Pi[:, m], '-', color=clr[m,:], label=f"mem {m}")
        axs.plot(0, P0[0, m], 'o', color=clr[m,:], markersize=10)

    for m in range(Nmem):
        axs.plot(ti, Pi[:, m+Nmem], ':', color=clr[m,:], label=f"ctrl {m}")
        axs.plot(0, P0[0, m+Nmem], 'o', color=clr[m,:], markersize=10)

    axs.set_xlabel('Time')
    axs.set_ylabel('Projection')
    axs.set_title('Projection on memory and control patterns')

    axs.legend(loc='lower right')

def plot_memory_and_control(scalefactor=5, toggle=1):
    """(Slide-37)"""

    # Network parameters
    N = 50
    tau = 1
    rmax = 150
    theta = -20
    Nmem = 4
    alpha = 0.2
    kappa = 2.0

    deltat = 0.02
    Tend = 5

    # make connectivity matrix
    (Vm, K, M, Cm_list) = ConnectivityMatrix( N, Nmem, alpha, kappa)

    # simulate dynamics
    # scaling of initial condition
    (ti, P0, Pi, Vi) = SimulateDynamics( N, Nmem, Vm, M, tau, deltat, Tend, scalefactor, rmax, theta, alpha, kappa )

    # Plot results
    fig, axs = plt.subplots( 1,1, figsize=(8,6))

    PlotProjectionDynamicWithControl( axs, Nmem, ti, P0, Pi)

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_memory_and_control, scalefactor = (1.0, 20.0, 1.0 ), toggle=(0,1,1) )


interactive(children=(FloatSlider(value=5.0, description='scalefactor', max=20.0, min=1.0, step=1.0), IntSlide…

<function __main__.plot_memory_and_control(scalefactor=5, toggle=1)>

# Avoiding overlap

Network performance becomes even more selective when overlap between memory patterns is avoided.  If this is the case, one pattern dominates and all others are suppressed.


In [ ]:
# @title No overlap between memory patterns {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import matplotlib.pyplot as plt

def MakeMemoryPattern(N, alpha):
    # length N, sparseness alpha
    # number on
    nc = round(alpha * N)
    # number off
    n0 = N - nc
    # memory vector
    vm = np.concatenate((np.ones(nc), np.zeros(n0)))
    # randomize order
    np.random.shuffle(vm)
    return vm

def FourPatternsNoOverlap(N, alpha):
    # length N, sparseness alpha
    # number on
    nc = round(alpha * N)
    # number off
    n0 = N - 4*nc
    # memory vectors
    Ones   = np.ones(nc)
    Twos   = 2 * np.ones(nc)
    Threes = 3 * np.ones(nc)
    Fours  = 4 * np.ones(nc)
    # memory vector
    vm = np.concatenate((Ones, Twos, Threes, Fours, np.zeros(n0)))
    # randomize order
    np.random.shuffle(vm)

    Vm = np.zeros((4, N))
    for m in range(4):
        ix = np.where(vm == m+1)
        Vm[m, ix] = 1;

    return Vm

def AutoCovarianceMatrix(vm, N, alpha):
    # pattern vm, length N, sparseness alpha
    v0 = vm - alpha * np.ones(N)
    return np.outer(v0, v0)

def ActivationFunction(vin, rmax, theta):
    # input vin, saturation rmax, threshold theta
    vout = rmax * np.tanh((vin - theta) / rmax)
    vout = np.maximum(vout, 0)
    return vout

def ConnectivityMatrixNoOverlap( N, Nmem, alpha, kappa):
    # size N, memories Nmem, sparseness alpha, recurrent excitation kappa
    Vm = np.empty((Nmem, N))
    K = np.zeros((N, N))
    Cm_list = []

    Vm = FourPatternsNoOverlap(N, alpha)

    for m in range(Nmem):
        vm = Vm[m,:]
        Cm = AutoCovarianceMatrix(vm, N, alpha)
        Cm_list.append(Cm)
        K += Cm

    K = K / ( alpha * (1 - alpha) * N )
    M = kappa * K - 1 / (alpha * N)

    return Vm, K, M, Cm_list


# Simulate dynamics
def SimulateDynamics( N, Nmem, Vm, M, tau, dt, Tend, scalefactor, rmax, theta, alpha, kappa ):

    # time axis
    ti = np.linspace(0, Tend, int(Tend / dt))

    # concatenate memory patterns and control patterns
    Fm = np.empty((2 * Nmem, N))
    Fm[:Nmem, :] = Vm
    for m in range(Nmem):
        Fm[Nmem + m, :] = MakeMemoryPattern(N, alpha)

    # initial condition as linear combination of two random memory patterns
    k1, k2 = np.random.randint(0, Nmem, 2)
    c1, c2 = scalefactor, 0
    vold   = c1 * Fm[k1, :] + c2 * Fm[k2, :]

    # projections of initial condition
    P0 = np.empty((1, 2 * Nmem))
    for m in range(2 * Nmem):
            fm = Fm[m, :]
            P0[0, m] = np.dot(vold, fm) / np.linalg.norm(fm)

    # allocate matrices for projections and for patterns
    Pi = np.empty((len(ti), 2 * Nmem))
    Vi = np.empty((len(ti), N))

    # iteration
    for i in range(len(ti)):
        vin = M @ vold
        vinf = ActivationFunction(vin, rmax, theta)
        vnew = vinf + (vold - vinf) * np.exp(-dt / tau)
        # new pattern
        Vi[i, :] = vnew
        vold = vnew
        # projections
        for m in range(2 * Nmem):
            fm = Fm[m, :]
            Pi[i, m] = np.dot(vnew, fm) / np.linalg.norm(fm)

    return ti, P0, Pi, Vi

def PlotProjectionDynamicWithControl( axs, Nmem, ti, P0, Pi):

    clr = np.array([[0.25, 0.75, 0.25],
                   [0.75, 0.25, 0.25],
                   [0.85, 0.85, 0.15],
                   [0.25, 0.25, 0.75]])

    for m in range(Nmem):
        axs.plot(ti, Pi[:, m], '-', color=clr[m,:], label=f"mem {m}")
        axs.plot(0, P0[0, m], 'o', color=clr[m,:], markersize=10)

    for m in range(Nmem):
        axs.plot(ti, Pi[:, m+Nmem], ':', color=clr[m,:], label=f"ctrl {m}")
        axs.plot(0, P0[0, m+Nmem], 'o', color=clr[m,:], markersize=10)

    axs.set_xlabel('Time')
    axs.set_ylabel('Projection')
    axs.set_title('Projection on non-overlapping memory patterns')

    axs.legend(loc='lower right')

def plot_memory_and_control(scalefactor=1, toggle=1):
    """(Slide-38)"""

    # Network parameters
    N = 50
    tau = 1
    rmax = 150
    theta = -20
    Nmem = 4
    alpha = 0.2
    kappa = 2.0

    deltat = 0.02
    Tend = 5

    # make connectivity matrix
    (Vm, K, M, Cm_list) = ConnectivityMatrixNoOverlap( N, Nmem, alpha, kappa)

    # simulate dynamics
    # scaling of initial condition
    (ti, P0, Pi, Vi) = SimulateDynamics( N, Nmem, Vm, M, tau, deltat, Tend, scalefactor, rmax, theta, alpha, kappa )

    # Plot results
    fig, axs = plt.subplots( 1,1, figsize=(8,6))

    PlotProjectionDynamicWithControl( axs, Nmem, ti, P0, Pi)

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_memory_and_control, scalefactor = (1.0, 20.0, 1.0 ), toggle=(0,1,1) )


interactive(children=(FloatSlider(value=1.0, description='scalefactor', max=20.0, min=1.0, step=1.0), IntSlide…

<function __main__.plot_memory_and_control(scalefactor=1, toggle=1)>

# Basins of attraction

Network trajectories from random initial conditions towards one of the four memory patterns.  Location in state space represents proximity to each pattern.   The central region contains states that may evolve to any of the pattern.  The corners contain states that reliably evolve to one pattern.  The latter regions may be considered 'basins of attraction'.

In [ ]:
# @title Basins of attraction {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import matplotlib.pyplot as plt


def RepeatedSimulations( N, Nmem, Vm, M, tau, deltat, Tend, scalefactor, rmax, theta, alpha, kappa, Nrepeat ):

    # time axis of simulation
    deltat = 0.1;
    Tend   = 15;
    ti = np.linspace(0, Tend, int(Tend/deltat));

    # allocate projections
    Pimk = np.zeros(( len(ti), Nmem, Nrepeat) );

    # loop over repeats
    for  nr in range(Nrepeat):

         # random initial condition

         vold = scalefactor * MakeMemoryPattern( N, alpha );

         # iterative dynamic
         for i in range(len(ti)):
             # some noise
             ni = 2*(np.random.rand(len(vold))) - 1.0;

             vin = M @ vold;

             vinf = ActivationFunction( vin, rmax, theta );

             vnew = vinf + ( vold - vinf ) * np.exp( -deltat/tau ) + ni;

             # get projections
             for m in range(Nmem):
                 vm  = Vm[m,:];

                 Pimk[ i, m, nr ] = np.dot( vnew,  vm) / np.linalg.norm( vm );

             vold = vnew;

    return ti, Pimk


# %% plot projections on vectors
#
# clr = 'rbgkmcy';
# fs = 16;
#
# figure;
# hold on;
# for m=1:Nmem
#     plot( ti, Pi(:,m), clr(m), 'LineWidth', 2 );
# end
# h = legend('v_1', 'v_2', 'v_3', 'v_4', 'Location', 'SouthEast' );
# set(h,'FontSize', fs );
#
# hold off;
# xlabel( 'time', 'FontSize', fs );
# ylabel( 'projection', 'FontSize', fs );



# plot trajectories in distance landscape
def PlotTrajectories( axs, ti, Pimk ):


    clr = np.array([[0.25, 0.75, 0.25],
                   [1.00, 0.25, 0.25],
                   [0.85, 0.85, 0.15],
                   [0.25, 0.25, 1.00]])


    fs = 16;

    Nrepeat = Pimk.shape[2]

    # assign patterns to four corners of virtual space
    # 1    2
    #
    # 3    4

    # projections represent PROXIMITY to corners, corner 3 is (0,0)

    # x and y coordinate are determined as weighted sums by proximity


    axs.fill( [-0.1, 1.1, 1.1, -0.1, -0.1], [-0.1, -0.1, 1.1, 1.1, -0.1], color='w' );

    for nr in range(Nrepeat):

        Pim = np.squeeze(Pimk[:,:,nr]);
        Pm = np.squeeze( Pim[-1,:] );

        # select color of largest projection
        kmax = np.argsort( Pm );
        kclr = kmax[-1];
        P1 = np.squeeze( Pim[:,0] );
        P2 = np.squeeze( Pim[:,1] );
        P3 = np.squeeze( Pim[:,2] );
        P4 = np.squeeze( Pim[:,3] );

        Pxi = (P2+P4) / (P1 + P2 + P3 + P4);
        Pyi = (P1+P2) / (P1 + P2 + P3 + P4);

        axs.plot( Pxi, Pyi, color=clr[kclr,:], linewidth=2 );


    epsilon = 0.05;
    axs.text( -epsilon, -epsilon, '3', fontsize=fs );
    axs.text( -epsilon, 1+epsilon, '1', fontsize=fs );
    axs.text( 1+epsilon, -epsilon, '4', fontsize=fs );
    axs.text( 1+epsilon, 1+epsilon, '2', fontsize=fs );

    axs.set_aspect(1/1);
    axs.set_xticks([0,1])
    axs.set_xticklabels(['',''])
    axs.set_yticks([0,1])
    axs.set_yticklabels(['',''])

def plot_basins_of_attraction(scalefactor=12, toggle=1):
    """(Slide-39)"""

    # Network parameters
    N = 50
    tau = 1
    rmax = 150
    theta = -20
    Nmem = 4
    alpha = 0.2
    kappa = 2.0

    deltat = 0.02
    Tend = 5

    # make connectivity matrix
    (Vm, K, M, Cm_list) = ConnectivityMatrixNoOverlap( N, Nmem, alpha, kappa)

    Nrepeat = 100;

    # simulate dynamics
    # scaling of initial condition
    (ti, Pimk) = RepeatedSimulations( N, Nmem, Vm, M, tau, deltat, Tend, scalefactor, rmax, theta, alpha, kappa, Nrepeat )

    # Plot results
    fig, axs = plt.subplots( 1,1, figsize=(8,6))

    PlotTrajectories( axs, ti, Pimk );

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_basins_of_attraction, scalefactor = (1.0, 20.0, 1.0 ), toggle=(0,1,1) )


interactive(children=(FloatSlider(value=12.0, description='scalefactor', max=20.0, min=1.0, step=1.0), IntSlid…

<function __main__.plot_basins_of_attraction(scalefactor=12, toggle=1)>

# Final thoughts

A recurrent network performs as an associative memory if the connectivity propagates excitation preferentially {\it within} memory patterns, so that memory patterns are stable fixed points of the dynamics.  


This works only if (i) the memory patterns are sparse ($\alpha << 1$) and (ii) there are far fewer patterns than units
($N_{mem} << N_v$).


If these conditions are violated, too much excitation 'leaks' from one pattern to other patterns so that the memory function is lost.


The covariance prescription used here is not optimal, and more efficient methods can be devised.

(Slide-40)

# **Overall summary**

- We have introduced the principle of 'content-addressable' or 'associative' memory.

- A recurrent network can perform this function, provided the desired memory patterns are fixed points of its activity.

- A simple way to achieve this is to base connectivity on the covariances of the desired memory patterns.

- In addition, the sigmoidal activation function must be just right (neither too steep, nor too flat).

- When initiated with an arbitrary activity pattern, such a recurrent network settles on the memory pattern that most resembles the input.

- Intuitively, we can visualize this in terms of a 'potential  landscape'.


<br>

# **Next:**

# **Lecture 5 State space analysis**


# **Appendix: justification of potential landscape**

Recall from Lecture 1 alternative formulations of firing rate model.


Synaptic current approaches steady-state more quickly:

$$
\tau_r \frac{dv(t)}{dt} = -\underbrace{v(t)}_\mathit{output\,activity} + F\left[ \underbrace{I_s(t)}_\mathit{synaptic\,current} \right], \qquad I_s \simeq w \underbrace{u(t)}_\mathit{input\, activity}
$$

Activity approaches steady-state more quickly:

$$
\tau_s \frac{dI_s(t)}{dt} = -\underbrace{I_s(t)}_\mathit{synaptic\,current} + w \underbrace{u(t)}_\mathit{input\,activity}, \qquad  \underbrace{v(t)}_\mathit{output\, activity} \simeq F\left[ I_s \right]
$$

To justify the 'potential landscape' metaphor, we choose the second formulation, in terms of synaptic input current ${\bf I}_s$, rather than output firing ${\bf v}$.


Recurrent network with output activity ${\bf v}$, synaptic input current ${\bf I}_s$, synaptic connectivity matrix ${\bf M}$, and feedforward input ${\bf h}$

$$
\tau_s \frac{d{\bf I}_s}{dt} = -{\bf I}_s + {\bf h} + {\bf M} \cdot {F}({\bf I}_s), \qquad {\bf v} = {F}( {\bf I}_s)
$$


Dynamic state variable is ${\bf I}_s(t)$.  Output activity follows as ${\bf v} = F({\bf I}_s)$.


# Lyapunov function

Assuming a symmetric ${\bf M}$ with zero diagonal, we formulate an expression $L_a(I_a ,h_a)$ for each component $h_a$, $I_a$:

$$
L_a(I_a,h_a) = \int_0^{I_a} \, i_a \, F'(i_a) \, di_a - h_a \, F(I_a) - \frac{1}{2} \, \sum_{a'} \, F(I_a) \, M_{aa'} \, F(I_{a'})
$$

where $F'(i) = \frac{dF(i)}{di}$.   Summing these expressions over all components, we obtain the **Lyapunov function**
$$
L({\bf I}_s) = \sum_a \, L_a(I_a,h_a)
$$

As long as activity ${\bf I}_s$ changes with time, the Lyapunov function decreases monotonically:

$$
\frac{d {\bf I}_s}{dt} \neq 0 \qquad \Rightarrow \qquad \frac{d{\bf L}}{dt} < 0
$$

# Time derivative

The time-derivative of expression $L_a$:

$$\begin{eqnarray}
\frac{dL_a}{dt} &= I_a \, F'(I_a) \, \frac{dI_a}{dt} - h_a \, F'(I_a) \,  \frac{dI_a}{dt}-
\\
&- \frac{1}{2}\frac{dI_a}{dt}  \sum_{a'} \, F'(I_a) \, M_{aa'} \, F(I_{a'}) - \frac{1}{2}  \sum_{a'} \, F(I_a) \, M_{aa'} \, F'(I_{a'}) \, \frac{dI_{a'}}{dt}
\end{eqnarray}$$

$$\begin{eqnarray}
\frac{dL}{dt} &= \sum_a \frac{dL_a}{dt} =
\\
&= \sum_a (I_a-h_a) F'(I_a) \frac{dI_a}{dt} - \sum_{aa'} \, F'(I_{a}) \, M_{aa'} \, F(I_{a'}) \, \frac{dI_{a}}{dt}=
\\
&= - \sum_a F'(I_a) \frac{dI_a}{dt} \left[ -I_a + h_a + \sum_{a'} \, M_{aa'} \, F(I_{a'}) \right] =
\\
&= - \frac{1}{\tau_s}  \sum_a F'(I_a) \left(\frac{dI_a}{dt}\right)^2  \quad \stackrel{!}{<} \quad 0
\end{eqnarray}$$

where we have used
$$
\tau_s \frac{dI_a}{dt} = -I_a + h_a + \sum_{a'} M_{aa'} \, F(I_{a'})
$$

$$
\frac{dL({\bf I})}{dt} = - \frac{1}{\tau_s}  \sum_a F'(I_a) \left(\frac{dI_a}{dt}\right)^2
$$

Since $F'(I_a)$ is monotonically increasing, the derivative is $>0$ unless and until $\frac{dI_a}{dt}=0$.


If $L({\bf I}_s)$ has a lower bound, then it cannot decrease indefinitely, so that ${\bf I}_s$ must converge to a fixed point.


Since
$$
{\bf I}_s = {\bf h} + {\bf M} \cdot {\bf v}
$$
this means that ${\bf v}$ converges to a fixed point, too.
